In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import torch
import torch.nn as nn

# Couche de convolution
Conv2d + BatchNorm2d + SiLU

In [3]:
class Conv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, groups=1, activation=True):
        super().__init__()
        self.conv=nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False, groups=groups)
        self.bn=nn.BatchNorm2d(out_channels, eps=0.001, momentum = 0.03)
        self.act=nn.SiLU(inplace=True) if activation else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

        
# epsilon = eps

# Bloc C2F (Cross-stage partial bottleneck with 2 convolutions)
Conv + Bottleneck + Conv

Combine les caractéristique de haut niveau avec les information contextuels pour améliorer la précision de la détection.
Avec les connexions residuelles



In [5]:
# Bottleneck : stack de 2 conv avec des connexions résuduelles (True ou  False)
class Bottleneck(nn.Module):
    def __init__(self, in_channels, out_channels, shortcut=True):
        super().__init__()
        self.conv1=Conv(in_channels, out_channels, kernel_size=3, stride = 1, padding=1)
        self.conv2=Conv(out_channels, out_channels, kernel_size=3, stride = 1, padding=1)
        self.shortcut=shortcut

    def forward(self, x):
        x_in = x # copy for residual connection
        x = self.conv1(x)
        x = self.conv2(x)
        if self.shortcut :
            x = x + x_in
        return x

class C2F(nn.Module):
    def __init__(self, in_channels, out_channels, num_bottlenecks, shortcut=True):
        super().__init__()
        self.mid_channels = out_channels//2
        self.num_bottlenecks = num_bottlenecks

        self.conv1 = Conv(in_channels, out_channels, kernel_size=1, stride=1, padding=0)

        # Séquence de Bottleneck
        self.m = nn.ModuleList([Bottleneck(self.mid_channels, self.mid_channels) for _ in range(num_bottlenecks)])
        self.conv2 = Conv((num_bottlenecks+2)*out_channels//2, out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        x = self.conv1(x)
        # Split
        x1, x2 = x[:, :x.shape[1]//2, :, :], x[:, x.shape[1]//2:, :, :] # Moitié des channels de x => après conv sortie = [N, C, H, W] N nombre de batch, C Nb channels, Height, Width  
        # List des sorties x1 et x2
        outputs = [x1, x2] 

        # x1 passe par le bottleneck
        for i in range(self.num_bottlenecks):
            x1 = self.m[i](x1)
            outputs.insert(0,x1)
            
        outputs = torch.cat(outputs, dim = 1) # concaténation [N, 0.5 c_out(num_bottleneck+2),W,H)
        out = self.conv2(outputs)

        return out

# Test de fonctionnement
c2f = C2F(in_channels=64, out_channels = 128, num_bottlenecks=2)
print(f"{sum(p.numel() for p in c2f.parameters())/1e6} millions de paramètres (Bloc C2F)")

dummy_input = torch.rand((1,64,244,244))
dummy_input = c2f(dummy_input)
print("Output shape: ", dummy_input.shape)
            
        
        

0.18944 millions de paramètres (Bloc C2F)
Output shape:  torch.Size([1, 128, 244, 244])


# SPPF
Spatial pyramide pooling fast:
Conv + Maxpool2d + Conv
Trraite les caractéristiques à différentes échelles et les pool en une carte de caractéristique de taille fixe (fixed-size feature map)


In [12]:
class SPPF(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=5):
        # Kernel size => taille des maxpool
        super().__init__()
        hidden_channels = in_channels//2
        self.conv1=Conv(in_channels, hidden_channels, kernel_size=1, stride = 1, padding = 0)
        # Concaténer outputs des maxpool et entrer dans conv2
        self.conv2=Conv(4*hidden_channels, out_channels, kernel_size=1, stride = 1, padding = 0)
        # Maxpool à 3 différentes échelles
        self.m= nn.MaxPool2d(kernel_size= kernel_size, stride = 1, padding = kernel_size//2, dilation = 1, ceil_mode=False)

    def forward(self, x):
        x = self.conv1(x)
        # Maxpool
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)

        # Concatener
        y = torch.cat([x,y1,y2,y3], dim=1)

        # final conv

        y = self.conv2(y)

        return y

# Test de fonctionnement
sppf = SPPF(in_channels = 128, out_channels=512)
print(f"{sum(p.numel() for p in c2f.parameters())/1e6} millions de paramètres (Bloc SPPF)")


dummy_output = sppf(dummy_input)
print("Output shape: ", dummy_output.shape)
            


0.18944 millions de paramètres (Bloc SPPF)
Output shape:  torch.Size([1, 512, 244, 244])
